# Lesson 12 Lab — AWQ: Protecting Salient Weights in W4A16

**Puzzle:** Can activation statistics tell us which weight channels deserve more protection?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

AWQ studies which weight channels are salient under observed activations and protects them within a weight-only W4A16 deployment path.

### Core mechanism

Channel scaling can preserve the floating-point linear transform while changing how weight ranges are shared before INT4 rounding. Activation statistics guide the scale search because frequently excited channels can amplify small weight errors.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "12-awq"
device = require_cuda()
torch.manual_seed(2026 + 12)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Protecting more channels or searching more scales costs calibration time and may reduce compression or kernel regularity. Lower weight MAE does not guarantee lower language-model loss.

### What this code tests

The notebook freezes calibration activations, searches scaling strength, and chooses by held-out layer-output error rather than weight error.

**Experiment:** Search activation-aware per-channel scaling strengths for a toy W4A16 layer and compare output error with naive INT4.

**Declared evidence label:** `numerical-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
cal=torch.randn(1024,256,device=device); cal[:,::29]*=7; test=torch.randn(512,256,device=device); test[:,::29]*=7
w=torch.randn(192,256,device=device); ref=test@w.t(); rows=[]
for alpha in (0.0,0.25,0.5,0.75,1.0):
    importance=cal.abs().mean(0).clamp_min(1e-5).pow(alpha); scaled=w*importance
    _,_,dq=symmetric_quantize(scaled,bits=4,group_size=64); restored=dq/importance
    rows.append({"alpha":alpha,"heldout_error":error_metrics(ref,test@restored.t())})
best=min(rows,key=lambda r:r["heldout_error"]["rmse"])
result=base_result(12,"numerical-model"); result.update({"alpha_sweep":rows,"best_alpha":best["alpha"],
    "conclusion":"Activation-aware scaling changed held-out W4A16 output error; no production AWQ kernel was claimed."})


## 3. Inspect the evidence

Use held-out output error and a frozen search set. Weight-only mean error is not the optimization target.

### Acceptance and rollback gate

Separate search/calibration from held-out evaluation, report protected fraction and group size, and prove a W4A16 operator executed before making speed claims.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "alpha_sweep": [
    {
      "alpha": 0.0,
      "heldout_error": {
        "cosine": 0.99425644,
        "mae": 2.19978571,
        "max_abs": 11.98613548,
        "rmse": 2.77175641
      }
    },
    {
      "alpha": 0.25,
      "heldout_error": {
        "cosine": 0.99609649,
        "mae": 1.80269086,
        "max_abs": 10.36782646,
        "rmse": 2.27352023
      }
    },
    {
      "alpha": 0.5,
      "heldout_error": {
        "cosine": 0.99506319,
        "mae": 2.01121306,
        "max_abs": 12.63252926,
        "rmse": 2.5623045
      }
    },
    {
      "alpha": 0.75,
      "heldout_error": {
        "cosine": 0.98953247,
        "mae": 2.91586828,
        "max_abs": 21.55734253,
        "rmse": 3.74847484
      }
    },
    {
      "alpha": 1.0,
      "heldout_error": {
        "cosine": 0.97454381,
        "mae": 4.58608341,
        "max_abs": 32.47526932,
        "rmse": 5.89838266
      }
    }
  ],
  "best_alpha": 0.25,
  "conclusion": "Activation-aware scaling 

## 4. Explain the result

Activation-aware protection is a model-quality method; deployment speed still requires a compatible W4A16 kernel.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).